# Data Quality Checks

Run this whenever new data lands (a new season backfill, or a weekly live-season update). Each section checks something we've actually been bitten by, or a structural invariant that should always hold. Anything that prints a non-empty result below is worth investigating before trusting downstream analysis.

In [1]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("mysql+pymysql://hockey_stats:hockey_stats@db:3306/hockey_stats")
pd.set_option("display.max_rows", 100)

## 1. Row counts per season

Sanity totals. Historic seasons (2023-24 onward) should each have several thousand events; 2022-23 has zero by design (source API has no event data for that season — confirmed directly, not a bug). The live 2026-27 season will show 0 events/players until matches start being played.

In [2]:
pd.read_sql("""
    SELECT
        s.name AS season,
        COUNT(DISTINCT m.id) AS matches,
        COUNT(DISTINCT e.id) AS events,
        COUNT(DISTINCT p.id) AS distinct_players
    FROM seasons s
    LEFT JOIN matches m ON m.season_id = s.id
    LEFT JOIN match_events e ON e.match_id = m.id
    LEFT JOIN players p ON p.id = e.player_id
    GROUP BY s.name
    ORDER BY s.name DESC
""", engine)

,season,matches,events,distinct_players
0,2026/27,1213,0,0
1,2025/26,1309,8916,2258
2,2024/25,1311,9204,2279
3,2023/24,1321,8338,2230
4,2022/23,1315,0,0


## 2. Match status codes — only expected values should appear

`R` = Result (completed), `A` = Abandoned, `S` = Scheduled (live season, not yet played), `P` = Postponed (rare — 1 seen across all 4 historic seasons). Anything else here means the source API returned a status we haven't seen before and haven't accounted for — check it before filtering on `status = 'R'` elsewhere.

In [3]:
status_counts = pd.read_sql("""
    SELECT s.name AS season, m.status, COUNT(*) AS matches
    FROM matches m
    JOIN seasons s ON s.id = m.season_id
    GROUP BY s.name, m.status
    ORDER BY s.name DESC, m.status
""", engine)

unexpected = status_counts[~status_counts["status"].isin(["R", "A", "S", "P"])]
print(f"Unexpected status codes: {len(unexpected)}")
if len(unexpected):
    display(unexpected)

status_counts

Unexpected status codes: 0


,season,status,matches
0,2026/27,S,1213
1,2025/26,A,1
2,2025/26,R,1308
3,2024/25,A,3
4,2024/25,R,1308
5,2023/24,A,8
6,2023/24,P,1
7,2023/24,R,1312
8,2022/23,A,3
9,2022/23,R,1312


## 3. Event types — only expected values should appear

Goal-type events: `FG` (field goal), `PC` (penalty corner goal — NOT an award, confirmed by cross-checking event counts against final scores), `PS` (penalty stroke). Card events: `GC`/`YC`/`RC` (green/yellow/red). Anything outside this set is new and needs the `card` mapping (in `eh_api.py`) extended.

In [4]:
event_types = pd.read_sql("""
    SELECT event_type, card, COUNT(*) AS n
    FROM match_events
    GROUP BY event_type, card
    ORDER BY n DESC
""", engine)

known = {"FG", "PC", "PS", "GC", "YC", "RC"}
unexpected = event_types[~event_types["event_type"].isin(known)]
print(f"Unexpected event types: {len(unexpected)}")
if len(unexpected):
    display(unexpected)

event_types

Unexpected event types: 0


,event_type,card,n
0,FG,NaN,9740
1,GC,green,7396
2,PC,NaN,5975
3,YC,yellow,2452
4,PS,NaN,870
5,RC,red,25


## 4. Team name hygiene

Two things to watch for here, both already investigated once:

- **Whitespace**: fixed for real clubs (Fareham, Loughborough Students). "TBC 1 Men"/"TBC 2 Men" will still show up — they're not real teams, just placeholders for one Abandoned 3rd/4th playoff match, already excluded by `status='R'` elsewhere. Harmless, left as-is.
- **Same name, different `id`**: after fixing a source-label bug that used to cause mass false duplication (a team/club/venue fetched by the live-season importer got a different `source` tag than the historic importer, so the DB's `(source, source_id)` uniqueness check didn't recognize them as the same row), the ~18 that remain are genuine — certain clubs don't add a gender suffix to their team name, so their real Men's and Women's teams share one display string (different `source_id` each). Not a bug; don't join by name alone across genders for these clubs.

In [5]:
teams = pd.read_sql("SELECT id, name FROM teams", engine)

whitespace_issues = teams[teams["name"] != teams["name"].str.strip()]
print(f"Team names with leading/trailing whitespace: {len(whitespace_issues)}")
if len(whitespace_issues):
    display(whitespace_issues)

teams["_norm"] = teams["name"].str.strip().str.lower()
dupes = teams[teams.duplicated("_norm", keep=False)].sort_values("_norm")
print(f"\nPossible duplicate teams (same name, different id): {dupes['_norm'].nunique()} name(s) affected")
if len(dupes):
    display(dupes.drop(columns="_norm"))

Team names with leading/trailing whitespace: 2


,id,name
175,433,TBC 1 Men
176,434,TBC 2 Men



Possible duplicate teams (same name, different id): 18 name(s) affected


,id,name
156,157,Bedford 1
164,293,Bedford 1
163,292,Beeston 2
157,158,Beeston 2
170,299,Belper 1
162,291,Belper 1
115,116,Cambridge City 1
148,149,Cambridge City 1
58,59,Chelmsford 1
183,441,Chelmsford 1


## 5. Score vs. event consistency

For completed matches (`status = 'R'`), the count of goal-type events (FG+PC+PS) should equal `home_score + away_score`. 2022-23 will show ~100% mismatch by design (zero event data that whole season). **2023-24 also has a real gap** (187 mismatches, confirmed against the live API — some matches there have partial or zero event logging, since it was the first season the digital event system existed and adoption was inconsistent). 2024-25 and 2025-26 are much cleaner (17 and 5 respectively) — treat any remaining mismatch there as worth a fresh look, since that season's data is otherwise mature.

In [6]:
mismatch = pd.read_sql("""
    SELECT
        s.name AS season,
        m.id AS match_id,
        m.home_score + m.away_score AS expected_goals,
        COUNT(e.id) AS actual_goal_events
    FROM matches m
    JOIN seasons s ON s.id = m.season_id
    LEFT JOIN match_events e ON e.match_id = m.id AND e.card IS NULL
    WHERE m.status = 'R'
    GROUP BY m.id, s.name, m.home_score, m.away_score
    HAVING expected_goals != actual_goal_events
""", engine)

print(f"Completed matches with a score/event mismatch: {len(mismatch)}")
mismatch.groupby("season").size().rename("mismatched_matches")

Completed matches with a score/event mismatch: 1492


season
2022/23    1283
2023/24     187
2024/25      17
2025/26       5
Name: mismatched_matches, dtype: int64